In [1]:
import time
import csv
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_total_params(model):
    return sum(p.numel() for p in model.parameters())

def get_peak_memory():
    return torch.cuda.max_memory_allocated() / 1024**2  # MB

os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
csv_file = "results/ResNet18_metrics.csv"


if not os.path.exists(csv_file):
    with open(csv_file, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "method", "epoch",
            "train_loss", "train_acc",
            "val_loss", "val_acc",
            "epoch_time_sec",
            "peak_memory_mb",
            "trainable_params"
        ])

## Loading dataset

In [3]:
# Pretrained EfficientNetV2-S weights
weights = ResNet18_Weights.IMAGENET1K_V1



# Use simple preprocessing only
transform = weights.transforms()

# Load Food101
train_dataset_full = datasets.Food101(
    root="./data",
    split="train",
    download=True,
    transform=transform
)

test_dataset = datasets.Food101(
    root="./data",
    split="test",
    download=True,
    transform=transform
)

# Split training set into train + validation
train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=8,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=8,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=8,
    pin_memory=True
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Train samples: 68175
Val samples: 7575
Test samples: 25250


In [4]:
method_name = "batchnorm_finetune"

# Load pretrained model
model = models.resnet18(weights=weights)

# get input features of final layer
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 101)
for param in model.parameters():
    param.requires_grad = False

# Unfreeze BatchNorm layers
for module in model.modules():
    if isinstance(module, nn.BatchNorm2d):
        for param in module.parameters():
            param.requires_grad = True
        module.train()  # allow BN running stats to update

# Tune ONLY the final classifier (ResNet uses fc)
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)
# print(model)

In [5]:
criterion = nn.CrossEntropyLoss()

## optimizeer for the batch norm and the classifier
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc

In [6]:
num_epochs = 8
best_val_acc = 0.0

trainable_params = count_trainable_params(model)
print(f"Trainable params: {trainable_params:,}")

for epoch in range(num_epochs):

    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    epoch_time = time.time() - start_time
    peak_memory = get_peak_memory()

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
    print(f"Time: {epoch_time:.2f}s | Peak Mem: {peak_memory:.2f} MB")
    print("-" * 40)

    # SAVE METRICS TO CSV
    with open(csv_file, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            method_name,
            epoch + 1,
            train_loss,
            train_acc,
            val_loss,
            val_acc,
            epoch_time,
            peak_memory,
            trainable_params
        ])

    # SAVE BEST MODEL
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        model_path = f"models/{method_name}_best.pth"
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
            "epoch": epoch
        }, model_path)

        print("Best model saved!")

Trainable params: 61,413
Epoch 1/8
Train Loss: 2.3097, Train Acc: 0.4548
Val Loss:   1.7283, Val Acc:   0.5696
Time: 1079.62s | Peak Mem: 2834.65 MB
----------------------------------------
Best model saved!
Epoch 2/8
Train Loss: 1.5578, Train Acc: 0.6009
Val Loss:   1.5737, Val Acc:   0.6011
Time: 1066.14s | Peak Mem: 2834.65 MB
----------------------------------------
Best model saved!
Epoch 3/8
Train Loss: 1.4134, Train Acc: 0.6349
Val Loss:   1.5245, Val Acc:   0.6129
Time: 1063.27s | Peak Mem: 2834.65 MB
----------------------------------------
Best model saved!
Epoch 4/8
Train Loss: 1.3365, Train Acc: 0.6516
Val Loss:   1.5024, Val Acc:   0.6194
Time: 1064.99s | Peak Mem: 2834.65 MB
----------------------------------------
Best model saved!
Epoch 5/8
Train Loss: 1.2850, Train Acc: 0.6636
Val Loss:   1.5197, Val Acc:   0.6194
Time: 1067.91s | Peak Mem: 2834.65 MB
----------------------------------------
Epoch 6/8
Train Loss: 1.2423, Train Acc: 0.6760
Val Loss:   1.4857, Val Acc:  